In [17]:
# ============================================
# Etiquetado manual de drift (Plotly + ipywidgets, simple)
# - Series_Ajustadas.csv con 'date_time' + N numéricas
# - Controles: DatePickers + HH:MM:SS + nudges ±1/±5/±10 min
# - ÚNICO CSV de trabajo: synthetic_data/etiquetado_manual.csv
#   (date_time,variable,event) con event ∈ {start,end}
# ============================================

from pathlib import Path
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import ipywidgets as W
from IPython.display import display
import re

# ------------------ Configuración ------------------
SERIES_PATH = Path("synthetic_data/synthetic_plant.csv")   # o "Series_Generadas/Series_Ajustadas.csv"
EVENTS_CSV  = Path("synthetic_data/etiquetado_manual.csv") # único archivo de trabajo (eventos)

# ------------------ Carga de datos ------------------
def detect_time_col(df: pd.DataFrame) -> str:
    for name in ["date_time","datetime","timestamp","time","fecha","tiempo"]:
        if name in df.columns:
            return name
    c0 = df.columns[0]
    pd.to_datetime(df[c0], errors="raise")
    return c0

assert SERIES_PATH.exists(), f"No encuentro {SERIES_PATH}"
raw = pd.read_csv(SERIES_PATH)
tcol = detect_time_col(raw)
df = raw.rename(columns={tcol: "date_time"})
df["date_time"] = pd.to_datetime(df["date_time"], errors="coerce")
df = df.dropna(subset=["date_time"]).set_index("date_time").sort_index()
num_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
df = df[num_cols].copy()
assert df.shape[1] > 0, "No hay columnas numéricas para etiquetar."

# ------------------ Estado/UI ------------------
current_col = df.columns[0]
episodes = []  # [{column, episode_start, episode_end}] SOLO en memoria para dibujar

# ------------------ Figura ------------------
fig = go.FigureWidget()
fig.update_layout(
    title="Etiquetado manual de drift",
    xaxis=dict(title="Tiempo", rangeslider=dict(visible=True)),
    yaxis=dict(title="Valor"),
    uirevision=True,
)
fig.add_scatter(x=df.index, y=df[current_col], mode="lines", name=current_col)

# ------------------ Utilidades ------------------
def _normalize_hhmmss(s: str) -> str:
    """Normaliza 'H', 'H:M' o 'H:M:S' a 'HH:MM:SS' (con límites)."""
    s = (s or "").strip()
    if not s:
        return "00:00:00"
    m = re.match(r"^\s*(\d{1,2})(?::(\d{1,2}))?(?::(\d{1,2}))?\s*$", s)
    if not m:
        return "00:00:00"
    h = int(m.group(1) or 0)
    mi = int(m.group(2) or 0)
    se = int(m.group(3) or 0)
    h = max(0, min(23, h)); mi = max(0, min(59, mi)); se = max(0, min(59, se))
    return f"{h:02d}:{mi:02d}:{se:02d}"

def refresh_plot():
    """Redibuja rectángulos de la columna actual."""
    with fig.batch_update():
        fig.layout.shapes = tuple([
            dict(type="rect", xref="x", yref="paper",
                 x0=r["episode_start"], x1=r["episode_end"], y0=0, y1=1,
                 fillcolor="rgba(200,60,60,0.25)",
                 line=dict(color="rgba(200,60,60,0.9)"),
                 layer="below")
            for r in episodes if r["column"] == current_col
        ])

def _merge_intervals_per_var(eps_df: pd.DataFrame) -> pd.DataFrame:
    """Une solapes/duplicados por variable (intervalos limpios)."""
    if eps_df.empty:
        return eps_df
    eps_df = (
        eps_df.drop_duplicates()
              .dropna(subset=["column","episode_start","episode_end"])
              .copy()
    )
    eps_df["episode_start"] = pd.to_datetime(eps_df["episode_start"], errors="coerce")
    eps_df["episode_end"]   = pd.to_datetime(eps_df["episode_end"],   errors="coerce")
    eps_df = eps_df.dropna(subset=["episode_start","episode_end"])
    eps_df = eps_df[eps_df["episode_end"] > eps_df["episode_start"]]
    out = []
    for col, g in eps_df.groupby("column", sort=True):
        g = g.sort_values(["episode_start","episode_end"]).reset_index(drop=True)
        if g.empty:
            continue
        cur_s = g.loc[0, "episode_start"]
        cur_e = g.loc[0, "episode_end"]
        for _, r in g.iloc[1:].iterrows():
            s, e = r["episode_start"], r["episode_end"]
            if s <= cur_e:
                if e > cur_e:
                    cur_e = e
            else:
                out.append({"column": col, "episode_start": cur_s, "episode_end": cur_e})
                cur_s, cur_e = s, e
        out.append({"column": col, "episode_start": cur_s, "episode_end": cur_e})
    return pd.DataFrame(out, columns=["column","episode_start","episode_end"])

def _episodes_to_events(eps_df: pd.DataFrame) -> pd.DataFrame:
    """Intervalos -> Eventos (date_time,variable,event in {'start','end'})."""
    if eps_df.empty:
        return pd.DataFrame(columns=["date_time","variable","event"])
    rows = []
    for _, r in eps_df.iterrows():
        v = r["column"]
        t0 = pd.to_datetime(r["episode_start"])
        t1 = pd.to_datetime(r["episode_end"])
        rows.append({"date_time": t0, "variable": v, "event": "start"})
        rows.append({"date_time": t1, "variable": v, "event": "end"})
    ev = pd.DataFrame(rows)
    ev["date_time"] = pd.to_datetime(ev["date_time"])
    ev = ev.drop_duplicates(subset=["date_time","variable","event"])
    return ev.sort_values(["variable","date_time","event"]).reset_index(drop=True)

def _events_to_episodes(ev_df: pd.DataFrame) -> pd.DataFrame:
    """Eventos -> Intervalos (empareja start/end por variable en orden temporal)."""
    need = {"date_time","variable","event"}
    if ev_df.empty or not need.issubset(set(ev_df.columns)):
        return pd.DataFrame(columns=["column","episode_start","episode_end"])
    g = ev_df.copy()
    g["date_time"] = pd.to_datetime(g["date_time"], errors="coerce")
    g = g.dropna(subset=["date_time","variable","event"])
    g = g.sort_values(["variable", "date_time", "event"])
    rows = []
    for var, grp in g.groupby("variable", sort=True):
        open_t = None
        for _, r in grp.iterrows():
            if str(r["event"]).lower() == "start":
                open_t = r["date_time"]
            elif str(r["event"]).lower() == "end" and open_t is not None and r["date_time"] > open_t:
                rows.append({"column": var, "episode_start": open_t, "episode_end": r["date_time"]})
                open_t = None
        # Si queda un start abierto sin end, lo ignoramos (o podrías cerrarlo al final del rango)
    eps = pd.DataFrame(rows)
    return _merge_intervals_per_var(eps) if not eps.empty else eps

# ------------------ Widgets ------------------
col_dd = W.Dropdown(options=list(df.columns), value=current_col, description="Serie:", layout=W.Layout(width="360px"))

date_ini = W.DatePicker(description="Inicio:")
hour_ini = W.Text(value="00:00:00", description="Hora:")
date_fin = W.DatePicker(description="Fin:")
hour_fin = W.Text(value="00:00:00", description="Hora:")

btn_add   = W.Button(description="Agregar intervalo", button_style="info")
btn_use   = W.Button(description="Usar selección → Agregar", tooltip="Toma lo que hay en los pickers y agrega")
btn_save  = W.Button(description="Guardar CSV", button_style="success")
btn_load  = W.Button(description="Cargar CSV", tooltip="Lee synthetic_data/etiquetado_manual.csv y reconstruye intervalos")
btn_clear = W.Button(description="Borrar de columna", button_style="warning", tooltip="Borra episodios visibles de la columna actual")

# Nudges ±1/±5/±10 min
btn_s_m10 = W.Button(description="⟵ ini -10m")
btn_s_m5  = W.Button(description="⟵ ini -5m")
btn_s_m1  = W.Button(description="⟵ ini -1m")
btn_s_p1  = W.Button(description="ini +1m ⟶")
btn_s_p5  = W.Button(description="ini +5m ⟶")
btn_s_p10 = W.Button(description="ini +10m ⟶")

btn_e_m10 = W.Button(description="⟵ fin -10m")
btn_e_m5  = W.Button(description="⟵ fin -5m")
btn_e_m1  = W.Button(description="⟵ fin -1m")
btn_e_p1  = W.Button(description="fin +1m ⟶")
btn_e_p5  = W.Button(description="fin +5m ⟶")
btn_e_p10 = W.Button(description="fin +10m ⟶")

out_msg = W.Output(layout=W.Layout(max_height="170px", overflow="auto", border="1px solid #444", padding="4px"))

# ------------------ Helper pickers ------------------
def _get_dt(date_widget, hour_widget):
    hh = _normalize_hhmmss(hour_widget.value)
    return pd.to_datetime(f"{date_widget.value} {hh}")

def _set_pickers(t0, t1):
    t0 = pd.to_datetime(t0); t1 = pd.to_datetime(t1)
    date_ini.value = t0.date()
    hour_ini.value = _normalize_hhmmss(t0.strftime("%H:%M:%S"))
    date_fin.value = t1.date()
    hour_fin.value = _normalize_hhmmss(t1.strftime("%H:%M:%S"))
    with out_msg: print(f"Prefill: {t0} → {t1}")

def _nudge(which, minutes):
    if which == "start":
        t = _get_dt(date_ini, hour_ini) + pd.Timedelta(minutes=minutes)
        date_ini.value = t.date(); hour_ini.value = _normalize_hhmmss(t.strftime("%H:%M:%S"))
    else:
        t = _get_dt(date_fin, hour_fin) + pd.Timedelta(minutes=minutes)
        date_fin.value = t.date(); hour_fin.value = _normalize_hhmmss(t.strftime("%H:%M:%S"))

# ------------------ Callbacks núcleo ------------------
def on_change_col(change):
    if change["name"] != "value":
        return
    set_series(change["new"])
    # Prefill: primer tramo de 1 hora
    _set_pickers(df.index.min(), df.index.min() + pd.Timedelta(hours=1))

def on_add(_):
    if not date_ini.value or not date_fin.value:
        with out_msg: print("⚠️ Debes elegir fechas de inicio y fin.")
        return
    t0 = _get_dt(date_ini, hour_ini)
    t1 = _get_dt(date_fin, hour_fin)
    if t1 <= t0:
        with out_msg: print("⚠️ El fin debe ser posterior al inicio.")
        return
    episodes.append({"column": current_col, "episode_start": t0, "episode_end": t1})
    refresh_plot()
    with out_msg: print(f"➕ Episodio agregado ({current_col}): {t0} → {t1}")

def on_use(_):
    on_add(_)

def on_save(_):
    # Episodios en memoria -> merge -> eventos -> guardar ÚNICO CSV
    eps_df = pd.DataFrame(episodes)
    eps_merged = _merge_intervals_per_var(eps_df) if not eps_df.empty else eps_df
    events_df = _episodes_to_events(eps_merged)
    events_df.to_csv(EVENTS_CSV, index=False)
    with out_msg:
        print(f"✅ Guardado eventos: {EVENTS_CSV} ({len(events_df)} filas)")

def on_load(_):
    global episodes
    if EVENTS_CSV.exists():
        try:
            ev = pd.read_csv(EVENTS_CSV)
            eps = _events_to_episodes(ev)
            episodes = eps.to_dict(orient="records")
            refresh_plot()
            with out_msg: print(f"📥 Cargados {len(episodes)} episodios reconstruidos desde {EVENTS_CSV}")
        except Exception as e:
            with out_msg: print(f"⚠️ Error al cargar eventos: {e}")
    else:
        with out_msg: print(f"⚠️ No existe {EVENTS_CSV}. Guarda primero desde la UI o crea el archivo.")

def on_clear(_):
    global episodes
    before = len(episodes)
    episodes = [r for r in episodes if r["column"] != current_col]
    removed = before - len(episodes)
    refresh_plot()
    with out_msg: print(f"🧹 Eliminados {removed} episodios de {current_col}")

def set_series(col):
    global current_col
    current_col = col
    with fig.batch_update():
        fig.data = ()
        fig.add_scatter(x=df.index, y=df[col], mode="lines", name=col)
    refresh_plot()

# ------------------ Enlaces y UI ------------------
col_dd.observe(on_change_col)
btn_add.on_click(on_add)
btn_use.on_click(on_use)
btn_save.on_click(on_save)
btn_load.on_click(on_load)
btn_clear.on_click(on_clear)

# Nudges INICIO
btn_s_m10.on_click(lambda _: _nudge("start", -10))
btn_s_m5 .on_click(lambda _: _nudge("start", -5))
btn_s_m1 .on_click(lambda _: _nudge("start", -1))
btn_s_p1 .on_click(lambda _: _nudge("start", +1))
btn_s_p5 .on_click(lambda _: _nudge("start", +5))
btn_s_p10.on_click(lambda _: _nudge("start", +10))
# Nudges FIN
btn_e_m10.on_click(lambda _: _nudge("end", -10))
btn_e_m5 .on_click(lambda _: _nudge("end", -5))
btn_e_m1 .on_click(lambda _: _nudge("end", -1))
btn_e_p1 .on_click(lambda _: _nudge("end", +1))
btn_e_p5 .on_click(lambda _: _nudge("end", +5))
btn_e_p10.on_click(lambda _: _nudge("end", +10))

# Inicialización
def set_series(col):
    global current_col
    current_col = col
    with fig.batch_update():
        fig.data = ()
        fig.add_scatter(x=df.index, y=df[col], mode="lines", name=col)
    refresh_plot()

set_series(current_col)
_set_pickers(df.index.min(), df.index.min() + pd.Timedelta(hours=1))

# Layout UI
row_top    = W.HBox([col_dd, btn_load, btn_save, btn_clear])
row_pickers= W.HBox([date_ini, hour_ini, date_fin, hour_fin, btn_add])
row_nudges_start = W.HBox([btn_s_m10, btn_s_m5, btn_s_m1, btn_s_p1, btn_s_p5, btn_s_p10])
row_nudges_end   = W.HBox([btn_e_m10, btn_e_m5, btn_e_m1, btn_e_p1, btn_e_p5, btn_e_p10])

ui = W.VBox([row_top, row_pickers, row_nudges_start, row_nudges_end, out_msg])
display(ui, fig)

print(f"Archivo único de eventos: {EVENTS_CSV.as_posix()}")


FigureWidget({
    'data': [{'mode': 'lines',
              'name': 'var_1',
              'type': 'scatter',
              'uid': '5d4e9e71-5093-43ef-bf88-56bcf0d72c1f',
              'x': array([datetime.datetime(2025, 1, 1, 0, 0),
                          datetime.datetime(2025, 1, 1, 1, 0),
                          datetime.datetime(2025, 1, 1, 2, 0), ...,
                          datetime.datetime(2025, 2, 6, 21, 0),
                          datetime.datetime(2025, 2, 6, 22, 0),
                          datetime.datetime(2025, 2, 6, 23, 0)], dtype=object),
              'y': array([ 0.01299983, -0.25908465, -0.77186411, ...,  1.09430768,  1.09583596,
                           1.30367568])}],
    'layout': {'template': '...',
               'title': {'text': 'Etiquetado manual de drift'},
               'uirevision': True,
               'xaxis': {'rangeslider': {'visible': True}, 'title': {'text': 'Tiempo'}},
               'yaxis': {'title': {'text': 'Valor'}}}
})

Archivo único de eventos: synthetic_data/etiquetado_manual.csv


In [ ]:
# ============================================
# Etiquetado manual de drift (Plotly + ipywidgets, simple)
# - Series_Ajustadas.csv con columna 'date_time' + N numéricas
# - Modos: selección (box), 2 clics, y CLICK SIMPLE → setea FECHA (nuevo)
# - Nudges finos: ±1/±5/±10 min para inicio y fin (corregidos)
# - Guardar/Cargar episodios por columna
# ============================================

from pathlib import Path
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import ipywidgets as W
from IPython.display import display
import re

# ------------------ Configuración ------------------
SERIES_PATH = Path("synthetic_data/synthetic_plant.csv") #"Series_Generadas/Series_Ajustadas.csv" - "synthetic_data/synthetic_plant.csv"
OUT_DIR = Path("labels_manual/Series_Ajustadas")
OUT_DIR.mkdir(parents=True, exist_ok=True)
MANUAL_CSV = Path("synthetic_data/etiquetado_manual.csv") #"Series_Generadas/etiquetado_manual.csv" - "synthetic_data/etiquetado_manual.csv"

# ------------------ Carga de datos ------------------
def detect_time_col(df: pd.DataFrame) -> str:
    for name in ["date_time","datetime","timestamp","time","fecha","tiempo"]:
        if name in df.columns:
            return name
    c0 = df.columns[0]
    pd.to_datetime(df[c0], errors="raise")
    return c0

assert SERIES_PATH.exists(), f"No encuentro {SERIES_PATH}"
raw = pd.read_csv(SERIES_PATH)
tcol = detect_time_col(raw)
df = raw.rename(columns={tcol: "date_time"})
df["date_time"] = pd.to_datetime(df["date_time"], errors="coerce")
df = df.dropna(subset=["date_time"]).set_index("date_time").sort_index()
num_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
df = df[num_cols].copy()
assert df.shape[1] > 0, "No hay columnas numéricas para etiquetar."

# ------------------ Estado global ------------------
current_col = df.columns[0]
episodes = []  # lista de dicts: {column, episode_start, episode_end}

# Si existe un CSV previo, cargamos
if MANUAL_CSV.exists():
    try:
        eps_prev = pd.read_csv(MANUAL_CSV, parse_dates=["episode_start","episode_end"])
        episodes = eps_prev.to_dict(orient="records")
    except Exception:
        episodes = []

# ------------------ Figura ------------------
fig = go.FigureWidget()
fig.update_layout(
    title="Etiquetado manual — Selección (box) / 2 clics / Click simple = fecha",
    xaxis=dict(title="Tiempo", rangeslider=dict(visible=True)),
    yaxis=dict(title="Valor"),
    uirevision=True,
)
fig.add_scatter(x=df.index, y=df[current_col], mode="lines", name=current_col)

# ------------------ Utilidades ------------------
def _normalize_hhmmss(s: str) -> str:
    """Normaliza entradas tipo 'H', 'H:M', 'H:M:S' a 'HH:MM:SS'."""
    s = (s or "").strip()
    if not s:
        return "00:00:00"
    m = re.match(r"^\s*(\d{1,2})(?::(\d{1,2}))?(?::(\d{1,2}))?\s*$", s)
    if not m:
        return "00:00:00"
    h = int(m.group(1) or 0)
    mi = int(m.group(2) or 0)
    se = int(m.group(3) or 0)
    h = max(0, min(23, h)); mi = max(0, min(59, mi)); se = max(0, min(59, se))
    return f"{h:02d}:{mi:02d}:{se:02d}"

def refresh_plot():
    """Redibuja los rectángulos de la columna actual."""
    with fig.batch_update():
        fig.layout.shapes = tuple([
            dict(type="rect", xref="x", yref="paper",
                 x0=r["episode_start"], x1=r["episode_end"], y0=0, y1=1,
                 fillcolor="rgba(200,60,60,0.25)",
                 line=dict(color="rgba(200,60,60,0.9)"),
                 layer="below")
            for r in episodes if r["column"] == current_col
        ])

def save_csv():
    if episodes:
        pd.DataFrame(episodes).sort_values(["column","episode_start"]).to_csv(MANUAL_CSV, index=False)

def load_for_column(col):
    eps = [r for r in episodes if r["column"] == col]
    if not eps:
        return pd.DataFrame(columns=["column","episode_start","episode_end"])
    return pd.DataFrame(eps)

def set_series(col):
    global current_col
    current_col = col
    with fig.batch_update():
        fig.data = ()
        fig.add_scatter(x=df.index, y=df[col], mode="lines", name=col)
    refresh_plot()

# ------------------ Widgets ------------------
col_dd = W.Dropdown(options=list(df.columns), value=current_col, description="Serie:", layout=W.Layout(width="360px"))

date_ini = W.DatePicker(description="Inicio:")
hour_ini = W.Text(value="00:00:00", description="Hora:")
date_fin = W.DatePicker(description="Fin:")
hour_fin = W.Text(value="00:00:00", description="Hora:")

btn_add   = W.Button(description="Agregar intervalo", button_style="info")
btn_use   = W.Button(description="Usar selección → Agregar", tooltip="Toma lo que hay en los pickers y agrega")
btn_save  = W.Button(description="Guardar CSV", button_style="success")
btn_load  = W.Button(description="Cargar CSV", tooltip="Vuelve a cargar del archivo")
btn_clear = W.Button(description="Borrar de columna", button_style="warning", tooltip="Borra episodios visibles de la columna actual")

btn_select_mode = W.ToggleButton(value=True, description="Seleccionar (box)")
btn_clicks_mode = W.ToggleButton(value=False, description="Modo 2 clics")
btn_single_click = W.ToggleButton(value=True, description="Click → set FECHA")  # NUEVO

# Nudges (corregidos) ±1/±5/±10 min
btn_s_m10 = W.Button(description="⟵ ini -10m")
btn_s_m5  = W.Button(description="⟵ ini -5m")
btn_s_m1  = W.Button(description="⟵ ini -1m")
btn_s_p1  = W.Button(description="ini +1m ⟶")
btn_s_p5  = W.Button(description="ini +5m ⟶")
btn_s_p10 = W.Button(description="ini +10m ⟶")

btn_e_m10 = W.Button(description="⟵ fin -10m")
btn_e_m5  = W.Button(description="⟵ fin -5m")
btn_e_m1  = W.Button(description="⟵ fin -1m")
btn_e_p1  = W.Button(description="fin +1m ⟶")
btn_e_p5  = W.Button(description="fin +5m ⟶")
btn_e_p10 = W.Button(description="fin +10m ⟶")

out_msg = W.Output(layout=W.Layout(max_height="170px", overflow="auto", border="1px solid #444", padding="4px"))

# ------------------ Helper pickers ------------------
def _get_dt(date_widget, hour_widget):
    hh = _normalize_hhmmss(hour_widget.value)
    return pd.to_datetime(f"{date_widget.value} {hh}")

def _set_pickers(t0, t1):
    t0 = pd.to_datetime(t0); t1 = pd.to_datetime(t1)
    date_ini.value = t0.date()
    hour_ini.value = _normalize_hhmmss(t0.strftime("%H:%M:%S"))
    date_fin.value = t1.date()
    hour_fin.value = _normalize_hhmmss(t1.strftime("%H:%M:%S"))
    with out_msg: print(f"Prefill: {t0} → {t1}")

def _nudge(which, minutes):
    if which == "start":
        t = _get_dt(date_ini, hour_ini) + pd.Timedelta(minutes=minutes)
        date_ini.value = t.date(); hour_ini.value = _normalize_hhmmss(t.strftime("%H:%M:%S"))
    else:
        t = _get_dt(date_fin, hour_fin) + pd.Timedelta(minutes=minutes)
        date_fin.value = t.date(); hour_fin.value = _normalize_hhmmss(t.strftime("%H:%M:%S"))

# ------------------ Callbacks núcleo ------------------
def on_change_col(change):
    if change["name"] != "value":
        return
    set_series(change["new"])
    # prefill por comodidad: primer tramo 1 hora
    _set_pickers(df.index.min(), df.index.min() + pd.Timedelta(hours=1))

def on_add(_):
    if not date_ini.value or not date_fin.value:
        with out_msg: print("⚠️ Debes elegir fechas de inicio y fin.")
        return
    t0 = _get_dt(date_ini, hour_ini)
    t1 = _get_dt(date_fin, hour_fin)
    if t1 <= t0:
        with out_msg: print("⚠️ El fin debe ser posterior al inicio.")
        return
    episodes.append({"column": current_col, "episode_start": t0, "episode_end": t1})
    refresh_plot()
    with out_msg: print(f"➕ Episodio agregado ({current_col}): {t0} → {t1}")

def on_use(_):
    on_add(_)

def on_save(_):
    save_csv()
    with out_msg: print(f"💾 Guardado en {MANUAL_CSV}")

def on_load(_):
    global episodes
    if MANUAL_CSV.exists():
        try:
            eps = pd.read_csv(MANUAL_CSV, parse_dates=["episode_start","episode_end"])
            episodes = eps.to_dict(orient="records")
            refresh_plot()
            with out_msg: print(f"📥 Cargado {len(episodes)} episodios desde {MANUAL_CSV}")
        except Exception as e:
            with out_msg: print(f"⚠️ Error al cargar CSV: {e}")
    else:
        with out_msg: print("⚠️ No existe aún el CSV para cargar.")

def on_clear(_):
    global episodes
    before = len(episodes)
    episodes = [r for r in episodes if r["column"] != current_col]
    removed = before - len(episodes)
    refresh_plot()
    with out_msg: print(f"🧹 Eliminados {removed} episodios de {current_col}")

# ------------------ Eventos del gráfico ------------------
# 1) Selección (box): arrastra para pre-rellenar pickers
def _on_select(trace, points, selector):
    if not btn_select_mode.value:
        return
    try:
        xs = np.array(points.xs) if hasattr(points, "xs") else None
    except Exception:
        xs = None
    if xs is None or xs.size == 0:
        idx = np.array(points.point_inds) if hasattr(points, "point_inds") else np.array([])
        if idx.size == 0:
            return
        xs = np.array(trace.x)[idx]
    t0, t1 = pd.to_datetime(xs.min()), pd.to_datetime(xs.max())
    if t1 <= t0:
        return
    _set_pickers(t0, t1)

# 2) Dos clics: primer clic = inicio, segundo = fin
_pending = []
def _on_click(trace, points, state):
    # Modo 2 clics
    if btn_clicks_mode.value:
        if not points.point_inds:
            return
        try:
            x = trace.x[points.point_inds[0]]
        except Exception:
            return
        _pending.append(pd.to_datetime(x))
        if len(_pending) == 1:
            with out_msg: print(f"[2 clics] Inicio provisional: {_pending[0]}")
        else:
            t0, t1 = sorted(_pending[:2]); _pending.clear()
            _set_pickers(t0, t1)
        return

    # CLICK SIMPLE → setea SOLO LA FECHA (conserva hh:mm actuales)
    if btn_single_click.value and points.point_inds:
        try:
            x = pd.to_datetime(trace.x[points.point_inds[0]])
        except Exception:
            return
        # Conserva horas/minutos de los pickers
        t0 = _get_dt(date_ini, hour_ini)
        t1 = _get_dt(date_fin, hour_fin)
        new_t0 = pd.Timestamp.combine(x.date(), t0.time())
        new_t1 = pd.Timestamp.combine(x.date(), t1.time())
        _set_pickers(new_t0, new_t1)
        with out_msg: print(f"[click] Fecha ajustada a {x.date()} (hh:mm conservados)")
        return

# Registrar (si el renderer lo soporta)
try:
    fig.data[0].on_selection(_on_select)
    fig.data[0].on_click(_on_click)
    with out_msg: print("Eventos de selección/2 clics/click simple activados.")
except Exception as e:
    with out_msg: print(f"Eventos no disponibles en este entorno: {e}")

# ------------------ Enlaces y UI ------------------
col_dd.observe(on_change_col)
btn_add.on_click(on_add)
btn_use.on_click(on_use)
btn_save.on_click(on_save)
btn_load.on_click(on_load)
btn_clear.on_click(on_clear)

# Nudges INICIO
btn_s_m10.on_click(lambda _: _nudge("start", -10))
btn_s_m5 .on_click(lambda _: _nudge("start", -5))
btn_s_m1 .on_click(lambda _: _nudge("start", -1))
btn_s_p1 .on_click(lambda _: _nudge("start", +1))
btn_s_p5 .on_click(lambda _: _nudge("start", +5))
btn_s_p10.on_click(lambda _: _nudge("start", +10))
# Nudges FIN
btn_e_m10.on_click(lambda _: _nudge("end", -10))
btn_e_m5 .on_click(lambda _: _nudge("end", -5))
btn_e_m1 .on_click(lambda _: _nudge("end", -1))
btn_e_p1 .on_click(lambda _: _nudge("end", +1))
btn_e_p5 .on_click(lambda _: _nudge("end", +5))
btn_e_p10.on_click(lambda _: _nudge("end", +10))

# Inicialización
set_series(current_col)
# Prefill inicial conveniente (primer tramo)
_set_pickers(df.index.min(), df.index.min() + pd.Timedelta(hours=1))

# Layout UI
row_top    = W.HBox([col_dd, btn_load, btn_save, btn_clear])
row_modes  = W.HBox([btn_select_mode, btn_clicks_mode, btn_single_click, btn_use])
row_pickers= W.HBox([date_ini, hour_ini, date_fin, hour_fin, btn_add])
row_nudges_start = W.HBox([btn_s_m10, btn_s_m5, btn_s_m1, btn_s_p1, btn_s_p5, btn_s_p10])
row_nudges_end   = W.HBox([btn_e_m10, btn_e_m5, btn_e_m1, btn_e_p1, btn_e_p5, btn_e_p10])

ui = W.VBox([row_top, row_modes, row_pickers, row_nudges_start, row_nudges_end, out_msg])
display(ui, fig)

print(f"CSV de salida: {MANUAL_CSV.as_posix()}")

FigureWidget({
    'data': [{'mode': 'lines',
              'name': 'var_1',
              'type': 'scatter',
              'uid': 'fdfb6f29-39e1-4c4b-b1af-8441d2aee2f9',
              'x': array([datetime.datetime(2025, 1, 1, 0, 0),
                          datetime.datetime(2025, 1, 1, 1, 0),
                          datetime.datetime(2025, 1, 1, 2, 0), ...,
                          datetime.datetime(2025, 2, 6, 21, 0),
                          datetime.datetime(2025, 2, 6, 22, 0),
                          datetime.datetime(2025, 2, 6, 23, 0)], dtype=object),
              'y': array([ 0.01299983, -0.25908465, -0.77186411, ...,  1.09430768,  1.09583596,
                           1.30367568])}],
    'layout': {'shapes': [{'fillcolor': 'rgba(200,60,60,0.25)',
                           'layer': 'below',
                           'line': {'color': 'rgba(200,60,60,0.9)'},
                           'type': 'rect',
                           'x0': Timestamp('2025-01-02 22:00:00'),


CSV de salida: synthetic_data/etiquetado_manual.csv
